## cat2num stats

bivariate category to numeric statistics --- is there a meaning statistical difference between means of categories.

t-test/anova:
- measure statistical significance difference bw means
- t-test: 2 categories
- anova (analysis of variance): 3 or more categories
- anova: indicates difference but not which one
- they do not measure correlation, unlike pearson r


T-TEST
1. t-score (-infinity, infinity) --- if + then 1st > 2nd group, if - then 2nd > 1st group
2. p-value (<0.05)
3. critical t

ANOVA

if f-score > crit f do tukey
1. f-score (0, infinity)
2. p-value (<0.05)
3. critical f
3. tukey hsd

In [12]:
# %conda install statsmodels
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

url = "https://raw.githubusercontent.com/dataprofessor/data/refs/heads/master/penguins_cleaned.csv"

df = pd.read_csv(url)
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181,3750,male
1,Adelie,Torgersen,39.5,17.4,186,3800,female
2,Adelie,Torgersen,40.3,18.0,195,3250,female
3,Adelie,Torgersen,36.7,19.3,193,3450,female
4,Adelie,Torgersen,39.3,20.6,190,3650,male


In [13]:
# T-TEST
males = df.loc[df["sex"] == "male", "body_mass_g"]
females = df.loc[df["sex"] == "female", "body_mass_g"]
t,p = stats.ttest_ind(males, females)
print(f"t={t:.4f}, p={p:4f}")

t=8.5417, p=0.000000


In [14]:
# getting critical t
deg_f = len(males) + len(females) - 2
alpha = 0.05
crit_t = stats.t.ppf(1 - alpha / 2, deg_f)
crit_t

np.float64(1.9671567996106814)

In [15]:
print("males on avg are bigger than females by this many grams")
print(males.mean() - females.mean())


males on avg are bigger than females by this many grams
683.4117965367964


In [16]:
# ANOVA
# this way is too slow
adelie = df.loc[df["species"] == "Adelie", "body_mass_g"]
gentoo = df.loc[df["species"] == "Gentoo", "body_mass_g"]
chinstrap = df.loc[df["species"] == "Chinstrap", "body_mass_g"]

f,p = stats.f_oneway(adelie, gentoo, chinstrap)
print(f"t={f:.4f}, p={p:4f}")

t=341.8949, p=0.000000


In [17]:
# lets do it faster
cols = df["species"].unique()
samples = [df.loc[df["species"] == var, "body_mass_g"] for var in cols]
f,p = stats.f_oneway(*samples)
print(f"f={f:.4f}, p={p:4f}")

f=341.8949, p=0.000000


In [18]:
# getting critical f
dfb = len(df["species"].unique()) - 1
dfw = len(df["body_mass_g"]) - len(df["species"].unique())
alpha = 0.05
crit_f = stats.f.ppf(1 - alpha, dfb, dfw)

print(f"critical={crit_f:.4f}")

critical=3.0231


In [19]:
scipy_tukey = stats.tukey_hsd(*samples)
print(scipy_tukey) #always print
cols


Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)  -1386.273     0.000 -1520.255 -1252.290
 (0 - 2)    -26.924     0.916  -186.201   132.353
 (1 - 0)   1386.273     0.000  1252.290  1520.255
 (1 - 2)   1359.349     0.000  1194.430  1524.267
 (2 - 0)     26.924     0.916  -132.353   186.201
 (2 - 1)  -1359.349     0.000 -1524.267 -1194.430



array(['Adelie', 'Gentoo', 'Chinstrap'], dtype=object)

- if p-value > 0.05 and the signs for CI are the same, we fail to reject Ho
- if p-value > 0.05 and the signs for CI are different, we fail to reject Ho
- if p-value < 0.05 and the signs for CI are different, we reject Ho
- if p-value < 0.05 and the signs for CI are the same, we reject Ho

In [20]:
sm_tukey = pairwise_tukeyhsd(
    endog = df["body_mass_g"], # numeric variable
    groups = df["species"],
    alpha = 0.05
)
sm_tukey.summary()
# this one has the second group > first when looking at meandiff

group1,group2,meandiff,p-adj,lower,upper,reject
Adelie,Chinstrap,26.9239,0.9164,-132.3528,186.2005,False
Adelie,Gentoo,1386.2726,0.0,1252.2897,1520.2554,True
Chinstrap,Gentoo,1359.3487,0.0,1194.4304,1524.2671,True
